<a href="https://colab.research.google.com/github/allaalmouiz/MachineTranslation_nlp/blob/main/MachineTranslation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Translation Challange - Module 5 | Part 2
Submitted by: **`Alaa Almouiz F. Moh.`**

ID Number: **`S2026_176`**

Track: **Machine Learning**

For: **ZAKA ©**

## **1- Problem Statement (Objective)**
Using  Neural Networks to build a model that translates text from **English to French**.

The model receives an English sentence as input and generates the corresponding French sentence as output. This is known as Neural Machine Translation (NMT).

### **Challenges:**
* Word **order** differs across languages.
* One word may **translate to several words**.
* **Context** affects meaning.
* Sentence **lengths vary**.

### **Expected model behaviour**
* The model may perform well on **short common sentences** - struggle with long sentences or rare words.
* The **vocabulary size may affec**t translation quality.

## **2. Loading the dataset**

In [1]:
!pip install -q nltk seaborn

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

In [3]:
# Loading the dataset from GitHub

!git clone https://github.com/allaalmouiz/MachineTranslation_nlp.git
%cd MachineTranslation_nlp


Cloning into 'MachineTranslation_nlp'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 15 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 2.37 MiB | 6.39 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/MachineTranslation_nlp


In [7]:
# Having the English and French datset
en = pd.read_csv("en.csv", header=None, names=["english"])
fr = pd.read_csv("fr.csv", header=None, names=["french"])

In [8]:
# Checking the first 5 samples of the English datset
en.head()

,english
0,"new jersey is sometimes quiet during autumn , ..."
1,the united states is usually chilly during jul...
2,"california is usually quiet during march , and..."
3,the united states is sometimes mild during jun...
4,"your least liked fruit is the grape , but my l..."


In [9]:
# Checking the first 5 samples of the English datset
fr.head()

,french
0,new jersey est parfois calme pendant l' automn...
1,les états-unis est généralement froid en juill...
2,"california est généralement calme en mars , et..."
3,"les états-unis est parfois légère en juin , et..."
4,"votre moins aimé fruit est le raisin , mais mo..."


In [15]:
# Printing the first english sentence and it's translation in french
print(f"English: {en['english'][0]}\nFrench: {fr['french'][0]}")
print("")
print(f"shape of the English datsets is: {en.shape}")
print(f"shape of the French datsets is: {fr.shape}")

English: new jersey is sometimes quiet during autumn , and it is snowy in april .
French: new jersey est parfois calme pendant l' automne , et il est neigeux en avril .

shape of the English datsets is: (137860, 1)
shape of the French datsets is: (137860, 1)


Now since we confirmed that they are in the same size, we can concatinate.

In [25]:
# Concatinate in one dataset and save into "en_fr_dataset.csv"
df = pd.concat([en, fr], axis=1)

df.to_csv("en_fr_dataset.csv", index=False)

In [26]:
df.head(1)


,english,french
0,"new jersey is sometimes quiet during autumn , ...",new jersey est parfois calme pendant l' automn...


In [27]:
# Checking null values

df.isnull().sum()

,0
english,0
french,0


## **3- Exploring and Processing the text dataset**

In [28]:
df["en_len"] = df["english"].apply(lambda x: len(x.split()))
df["fr_len"] = df["french"].apply(lambda x: len(x.split()))

print(df[["en_len","fr_len"]].describe())

              en_len         fr_len
count  137860.000000  137860.000000
mean       13.225374      14.226716
std         3.191240       3.016955
min         3.000000       3.000000
25%         9.000000      12.000000
50%        15.000000      15.000000
75%        15.000000      16.000000
max        17.000000      23.000000


In [32]:
print(f"The maximum sentence length for english is {df['en_len'].max()}")
print(f"The maximum sentence length for french is {df['fr_len'].max()}")

The maximum sentence length for english is 17
The maximum sentence length for french is 23


In [33]:
df.head()

,english,french,en_len,fr_len
0,"new jersey is sometimes quiet during autumn , ...",new jersey est parfois calme pendant l' automn...,15,16
1,the united states is usually chilly during jul...,les états-unis est généralement froid en juill...,17,15
2,"california is usually quiet during march , and...","california est généralement calme en mars , et...",15,15
3,the united states is sometimes mild during jun...,"les états-unis est parfois légère en juin , et...",16,15
4,"your least liked fruit is the grape , but my l...","votre moins aimé fruit est le raisin , mais mo...",16,16
